# Open Arena: Show-Me-How for a Constrained Research Agent

This notebook is a fast, business-first walkthrough of how Open Arena can evaluate a **Vanguard Research-style** agent.

The core use case is not generic question answering. It is a constrained deep-research workflow that turns a mission into a structured report with sources.

It is designed for a mixed audience:
- business stakeholders who want to understand why evaluation must reflect real work
- technical teammates who want to see how the pipeline is actually wired
- SME reviewers who care about whether the final report is scoped, defensible, and usable

**Best results:** run this notebook from the repo environment (the `.venv` created with `uv`) so the validation cells can import the real project code.

## 1. Why realistic mission-based evaluation matters

For a constrained research agent, evaluating with a few generic prompts is not enough.

The important question is:

**Can the system stay inside the requested domain, use the allowed sources, respect the time window, and produce a decision-useful report?**

That is why this demo uses **research missions** instead of support FAQs.

A realistic mission may require the agent to:
- stay on one technical theme instead of drifting into generic AI coverage
- treat allowed domains as a hard constraint
- respect a specific publication window
- synthesize evidence into a structured report with sources
- produce something useful for product, strategy, or innovation teams

## 2. The mental model for Open Arena

The simplest way to explain the repository is still:

**dataset -> experiments -> evaluation -> Langfuse**

That model is visible directly in the codebase:
- `README.md` explains the high-level workflow and project purpose
- `src/config/types.py` defines the config schema for dataset, experiments, and evaluation
- `src/main_cli.py` orchestrates loading rows, running experiments, evaluating results, and flushing traces to Langfuse

What changes in this refresh is the workload: instead of a generic assistant benchmark, we are evaluating a constrained research agent on mission briefs.

In [ ]:
from pathlib import Path
import csv
import os
import sys

from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
DEMO_DIR = REPO_ROOT / "demo" / "show_me_how_open_arena"
DATASET_PATH = DEMO_DIR / "data" / "business_qa_demo.csv"
SHOWCASE_CONFIG_PATH = DEMO_DIR / "configs" / "business_qa_showcase.yaml"
RUNNABLE_CONFIG_PATH = DEMO_DIR / "configs" / "business_qa_runnable.yaml"
ENV_PATH = REPO_ROOT / ".env"

load_dotenv(ENV_PATH)

{
    "repo_root": str(REPO_ROOT),
    "demo_dir": str(DEMO_DIR),
    "dataset_exists": DATASET_PATH.exists(),
    "showcase_config_exists": SHOWCASE_CONFIG_PATH.exists(),
    "runnable_config_exists": RUNNABLE_CONFIG_PATH.exists(),
    "env_file_exists": ENV_PATH.exists(),
}

## 3. What makes the Vanguard Research-style agent different

This demo is modeled on a research workflow where the agent transforms a mission into a structured report with citations.

The key behaviors are:
- **constrained retrieval** rather than open-ended browsing
- **mission-native retrieval** so the search stays intrinsic to the requested topic
- **allowed domains** treated as a real scope boundary
- **time windows** treated as hard constraints, not hints
- **final synthesis** shaped as a report, not just an answer snippet

That distinction matters for evaluation, because a seemingly fluent answer is not enough if it drifts off-topic or ignores the retrieval constraints.

In [ ]:
with DATASET_PATH.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

preview_fields = [
    "mission_title",
    "research_domain",
    "timeframe_start",
    "timeframe_end",
    "allowed_domains",
    "output_type",
]

print(f"Research missions in demo dataset: {len(rows)}")
[{key: row[key] for key in preview_fields} for row in rows[:3]]

## 4. What the mission dataset is showing

Each row is a compact mission brief for the agent.

The most important fields are:
- `mission_title` — human-readable framing of the task
- `research_domain` and `topic_cluster` — what area the mission belongs to
- `timeframe_start` and `timeframe_end` — the hard publication window
- `allowed_domains` — where the agent is allowed to source evidence from
- `focus_semantics` — the exact retrieval focus that should anchor the search
- `output_type` and `audience` — what shape the report should take and who it is for
- `expected_answer` — not a literal gold report, but the expected behavior and quality bar

This gives us a much more realistic benchmark for agent evaluation than a generic business QA sheet.

In [ ]:
print("=== Showcase config (for slides) ===")
print(SHOWCASE_CONFIG_PATH.read_text())
print("\n=== Runnable config (OpenAI backend under the hood) ===")
print(RUNNABLE_CONFIG_PATH.read_text())

In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config.types import ExperimentsFile

showcase_config = ExperimentsFile.from_yaml(SHOWCASE_CONFIG_PATH)
runnable_config_template = ExperimentsFile.from_yaml(RUNNABLE_CONFIG_PATH)

backend_model_map = [
    {
        "experiment_name": showcase_experiment.name,
        "showcase_model": showcase_experiment.litellm.model,
        "backend_model": runnable_experiment.litellm.model,
    }
    for showcase_experiment, runnable_experiment in zip(
        showcase_config.experiments,
        runnable_config_template.experiments,
        strict=True,
    )
]

{
    "showcase_dataset_name": showcase_config.dataset.name,
    "showcase_dataset_limit": showcase_config.dataset.limit,
    "showcase_models": [experiment.litellm.model for experiment in showcase_config.experiments],
    "runnable_dataset_name": runnable_config_template.dataset.name,
    "runnable_dataset_limit": runnable_config_template.dataset.limit,
    "backend_model_map": backend_model_map,
}

## 5. What the two configs are doing

This demo now uses **two YAML configs** on purpose.

- `business_qa_showcase.yaml` is the slide-friendly version with recent vendor model names from Gemini, Anthropic, Hugging Face, and OpenAI.
- `business_qa_runnable.yaml` keeps those same experiment labels, but the actual backend models are still OpenAI so the notebook can run locally without extra vendor setup.

That split is useful in a show-me-how format:
- one config is easy to show to stakeholders
- the other is reliable to execute in the notebook
- the backend mapping stays explicit instead of being hidden in prose

## 6. What happens at runtime

The runtime story is now intentionally split in two layers:

1. load the slide-friendly showcase YAML
2. load the runnable YAML that keeps the same experiment labels
3. cap the run at **20 rows per experiment**
4. launch the Open Arena CLI from the notebook
5. read back the dataset runs from Langfuse
6. build a score summary that clearly shows both the experiment label and the backend model

That makes the demo easier to explain live: the audience sees recognizable vendor labels, while the notebook still runs on a controlled local setup.

In [ ]:
required_env = [
    "LANGFUSE_SECRET_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_HOST",
    "OPENAI_API_KEY",
    "GEMINI_API_KEY",
    "ANTHROPIC_API_KEY",
    "HUGGINGFACE_API_KEY",
]

SHOWCASE_SAMPLE_LIMIT = 20

runtime_config = runnable_config_template.model_copy(deep=True)
runtime_config.dataset.limit = SHOWCASE_SAMPLE_LIMIT
runtime_config.dataset.name = f"{runnable_config_template.dataset.name} - sample {SHOWCASE_SAMPLE_LIMIT}"

runtime_row_count = min(len(rows), SHOWCASE_SAMPLE_LIMIT)
expected_experiment_calls = runtime_row_count * len(runtime_config.experiments)
expected_evaluation_calls = (
    runtime_row_count * len(runtime_config.experiments)
    if runtime_config.evaluation.method == "llm_as_judge"
    else runtime_row_count
)

{
    "env_status": {key: bool(os.getenv(key)) for key in required_env},
    "showcase_config_path": str(SHOWCASE_CONFIG_PATH),
    "runnable_config_path": str(RUNNABLE_CONFIG_PATH),
    "showcase_sample_limit": SHOWCASE_SAMPLE_LIMIT,
    "runtime_dataset_name": runtime_config.dataset.name,
    "runtime_row_count": runtime_row_count,
    "experiment_count": len(runtime_config.experiments),
    "expected_experiment_calls": expected_experiment_calls,
    "expected_evaluation_calls": expected_evaluation_calls,
    "backend_model_map": backend_model_map,
}

## 7. Langfuse local demo checklist

For this show-me-how, Langfuse is the place where the evaluation run becomes inspectable.

The notebook now assumes this split:
- **showcase config** with recent Gemini, Anthropic, Hugging Face, and OpenAI model labels
- **runnable config** with the same experiment names but OpenAI backends underneath

Environment checklist:
- `LANGFUSE_SECRET_KEY`
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_HOST`
- `OPENAI_API_KEY`
- `GEMINI_API_KEY` (demo placeholder is fine)
- `ANTHROPIC_API_KEY` (demo placeholder is fine)
- `HUGGINGFACE_API_KEY` (demo placeholder is fine)

For the live notebook run, the sample cap is fixed to **20 rows per experiment** so the comparison stays readable and does not take too long.

A good live path is:
1. show the showcase YAML
2. show the backend model mapping
3. run the notebook cell that launches the Open Arena workflow
4. inspect the Langfuse traces and final score summary

In [ ]:
hero_mission = rows[0]
hero_fields = [
    "mission_title",
    "research_domain",
    "timeframe_start",
    "timeframe_end",
    "allowed_domains",
    "focus_semantics",
    "output_type",
    "question",
    "expected_answer",
]

{key: hero_mission[key] for key in hero_fields}

## 8. What the final report is meant to look like

A good output should feel like a **Vanguard Research** deliverable, not like a loose browser summary.

A strong final report usually contains:
- an executive summary
- why the topic matters for the target audience
- the strongest signals, benchmarks, or workflow evidence
- a short list of implications or recommendations
- explicit uncertainty where the evidence is incomplete
- a final sources section

In other words, the benchmark is asking for a structured report with sources, not just a plausible paragraph.

## 9. Hero mission walkthrough

For the live demo, it is enough to anchor the audience on one hero mission.

A practical sequence is:
1. show the mission title and explain why it is realistic
2. point at the hard time window
3. point at the allowed domains
4. explain why the agent must stay close to the requested semantic focus
5. explain what kind of report the audience should expect at the end

That lets you connect the dataset row, the YAML prompt template, the Langfuse traces, and the idea of a final research report without drowning the audience in implementation detail.

## 10. Suggested 10-minute demo narration

Here is a practical live sequence:

- **0:00-1:00** — frame the hero mission and why it is a realistic request
- **1:00-2:30** — show the CSV row and explain the mission fields
- **2:30-4:00** — show the YAML config and explain how the mission brief is rendered
- **4:00-6:00** — explain the runtime flow in Open Arena
- **6:00-8:00** — inspect what Langfuse would make visible for the run
- **8:00-10:00** — close on what a good report looks like and why that makes the decision defensible

This keeps the demo operational and credible, without pretending to be a full product walkthrough.

## 11. Final takeaway

Open Arena is not only a benchmarking script.

For this use case, its value is that it turns a constrained research workload into something that is:
- explicitly declared
- comparable across variants
- observable in Langfuse
- evaluable against a clear quality bar

That is the shift from **"this answer looks good"** to **"this workflow is reviewable, reproducible, and defensible."**

In [ ]:
import shlex
import subprocess
import tempfile
from datetime import datetime, timedelta, timezone

import polars as pl
import yaml
from langfuse import get_client

try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    def display(value):
        print(value)

    def Markdown(value):
        return value

    def HTML(value):
        return value


def _truncate(text: str | None, limit: int = 180) -> str:
    clean = (text or "").replace("\n", " ").strip()
    if len(clean) <= limit:
        return clean
    return clean[: limit - 1] + "…"


def _render_score_cards(score_summary: pl.DataFrame):
    if score_summary.is_empty():
        return HTML("<p>No score summary available.</p>")

    cards = []
    for row in score_summary.to_dicts():
        fill = int(round(100 * (row["avg_score_0_to_1"] or 0.0)))
        cards.append(
            f"""
            <div style='border:1px solid #ddd;border-radius:10px;padding:14px;margin:10px 0;'>
              <div style='font-size:16px;font-weight:700;'>{row['experiment_name']}</div>
              <div style='color:#555;margin-top:2px;'>backend {row['model_name']}</div>
              <div style='margin-top:10px;font-size:26px;font-weight:700;'>{row['avg_score_1_to_5']}/5</div>
              <div style='margin-top:6px;background:#f1f1f1;border-radius:999px;height:12px;overflow:hidden;'>
                <div style='width:{fill}%;background:#2f7ed8;height:12px;'></div>
              </div>
              <div style='margin-top:10px;color:#555;'>
                scored {row['scored_items']} of {row['items']} items · min {row['min_score_1_to_5']}/5 · max {row['max_score_1_to_5']}/5
              </div>
            </div>
            """
        )
    return HTML("".join(cards))


def _load_run_traces(dataset_name: str, experiment_models: dict[str, str], run_started_at: datetime) -> pl.DataFrame:
    client = get_client()
    runs = client.api.datasets.get_runs(dataset_name=dataset_name, limit=max(40, len(experiment_models) * 10)).data

    matched_runs = {}
    for experiment_name in experiment_models:
        matching = [
            run
            for run in runs
            if run.created_at >= run_started_at - timedelta(seconds=5)
            and run.name.startswith(f"{experiment_name} - ")
        ]
        if matching:
            matched_runs[experiment_name] = max(matching, key=lambda run: run.created_at)

    if set(matched_runs) != set(experiment_models):
        missing = sorted(set(experiment_models) - set(matched_runs))
        raise RuntimeError(f"Missing Langfuse dataset runs for experiments: {missing}")

    trace_rows = []
    for experiment_name, run in matched_runs.items():
        run_details = client.api.datasets.get_run(dataset_name=dataset_name, run_name=run.name)
        for run_item in run_details.dataset_run_items:
            trace = client.api.trace.get(run_item.trace_id)
            matching_scores = [
                score for score in trace.scores if score.name == runtime_config.evaluation.score_name
            ]
            trace_score = matching_scores[0] if matching_scores else None
            trace_input = trace.input if isinstance(trace.input, dict) else {"input": str(trace.input or "")}
            trace_rows.append(
                {
                    "experiment_name": experiment_name,
                    "model_name": experiment_models[experiment_name],
                    "run_name": run.name,
                    "trace_id": trace.id,
                    "trace_url": os.environ["LANGFUSE_HOST"].rstrip("/") + trace.html_path,
                    "input_preview": _truncate(trace_input.get("input", ""), limit=140),
                    "output_preview": _truncate(str(trace.output or ""), limit=160),
                    "score_0_to_1": trace_score.value if trace_score else None,
                    "score_1_to_5": round(1 + 4 * trace_score.value, 2) if trace_score else None,
                    "judge_comment": _truncate(trace_score.comment if trace_score else "", limit=220),
                }
            )

    return pl.DataFrame(trace_rows)


backend_model_table = pl.DataFrame(backend_model_map)
experiment_models = {
    experiment.name: experiment.litellm.model
    for experiment in runtime_config.experiments
}

runtime_config_file = tempfile.NamedTemporaryFile(
    "w", suffix=".yaml", delete=False, encoding="utf-8"
)
runtime_config_path = Path(runtime_config_file.name)
yaml.safe_dump(
    runtime_config.model_dump(mode="python"),
    runtime_config_file,
    sort_keys=False,
    allow_unicode=True,
)
runtime_config_file.close()

run_command = [
    str(REPO_ROOT / ".venv" / "bin" / "python"),
    "-m",
    "src.main_cli",
    "--config",
    str(runtime_config_path),
]

run_started_at = datetime.now(timezone.utc)

display(
    Markdown(
        f"""
## 12. Live run and final score summary

This cell shows the slide-friendly config, the real backend mapping, and then launches the runnable Open Arena workflow.

- **Showcase config:** `{SHOWCASE_CONFIG_PATH.name}`
- **Runnable config:** `{RUNNABLE_CONFIG_PATH.name}`
- **Rows per experiment:** {SHOWCASE_SAMPLE_LIMIT}
- **Experiments:** {len(runtime_config.experiments)}
- **Experiment calls:** {expected_experiment_calls}
- **Evaluation calls:** {expected_evaluation_calls}

```bash
{' '.join(shlex.quote(part) for part in run_command)}
```
"""
    )
)

display(Markdown("### Showcase model → backend model"))
display(backend_model_table)

captured_output = []
try:
    process = subprocess.Popen(
        run_command,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout or []:
        print(line, end="")
        captured_output.append(line)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Open Arena run failed with exit code {return_code}. Last output: {''.join(captured_output[-20:])}"
        )

    display(Markdown("### Collecting Langfuse traces and score rows"))
    trace_df = _load_run_traces(
        runtime_config.dataset.name,
        experiment_models,
        run_started_at,
    )

    score_summary = (
        trace_df.group_by(["experiment_name", "model_name", "run_name"])
        .agg(
            [
                pl.len().alias("items"),
                pl.col("score_0_to_1").is_not_null().sum().alias("scored_items"),
                pl.col("score_0_to_1").mean().alias("avg_score_0_to_1"),
                pl.col("score_0_to_1").min().alias("min_score_0_to_1"),
                pl.col("score_0_to_1").max().alias("max_score_0_to_1"),
            ]
        )
        .with_columns(
            [
                (1 + 4 * pl.col("avg_score_0_to_1")).round(2).alias("avg_score_1_to_5"),
                (1 + 4 * pl.col("min_score_0_to_1")).round(2).alias("min_score_1_to_5"),
                (1 + 4 * pl.col("max_score_0_to_1")).round(2).alias("max_score_1_to_5"),
            ]
        )
        .sort(["avg_score_1_to_5", "experiment_name"], descending=[True, False])
    )

    display(Markdown("### Final score summary"))
    display(_render_score_cards(score_summary))
    display(
        score_summary.select(
            [
                "experiment_name",
                "model_name",
                "items",
                "scored_items",
                "avg_score_1_to_5",
                "avg_score_0_to_1",
                "min_score_1_to_5",
                "max_score_1_to_5",
            ]
        )
    )

    item_score_table = trace_df.select(
        [
            "experiment_name",
            "model_name",
            "score_1_to_5",
            "input_preview",
            "judge_comment",
            "trace_url",
        ]
    )

    display(Markdown("### Lowest-scored examples"))
    display(item_score_table.sort(["score_1_to_5", "experiment_name"]).head(12))

finally:
    runtime_config_path.unlink(missing_ok=True)